# Convergence tests for the cohomology proxy C(Z)

Purpose:
- Test numerical stability of C(Z) under cover refinement, integrator tolerances, and r_max.
- Produce stability plots and a small CSV summarizing results.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Ensure repo root is importable
repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Adjust imports to your solver API
try:
    from src.dirac_radial_solver import compute_E1s, compute_C_for_Z
except Exception as e:
    print('Warning: adjust imports to match your solver API:', e)


In [ ]:
# Parameters
Z_test = 36
model = 'sphere'
R_fm = 5.0
r_max_list = [200.0, 300.0, 400.0]
cover_n_list = [6, 8, 12]
rtol_list = [1e-6, 1e-8, 1e-10]
atol = 1e-10
eps_factor = 1e-6

out_dir = os.path.join('results','convergence')
os.makedirs(out_dir, exist_ok=True)
summary_rows = []


In [ ]:
def compute_C(Z, model='sphere', R_fm=5.0, r_max=300.0, cover_n=8, rtol=1e-8, atol=1e-10, eps_factor=1e-6):
    # Wrapper: adapt to your solver API
    try:
        E1s_au = compute_E1s(Z, model=model, R_fm=R_fm, r_max=r_max, rtol=rtol, atol=atol)
    except Exception as e:
        print('compute_E1s failed:', e)
        E1s_au = None
    try:
        sigma_vals = compute_C_for_Z(Z, model=model, R_fm=R_fm, r_max=r_max, cover_n=cover_n, rtol=rtol, atol=atol)
        sigma_vals = np.asarray(sigma_vals)
        sigma_max = float(np.max(sigma_vals))
        sigma_min = float(np.min(sigma_vals))
        eps = eps_factor * sigma_max
        C = int(np.sum(sigma_vals < eps))
    except Exception as e:
        print('compute_C_for_Z or SVD failed:', e)
        sigma_min = None
        sigma_max = None
        eps = None
        C = None
    return dict(Z=Z, E1s_au=E1s_au, E1s_eV=(E1s_au*27.211386) if E1s_au is not None else None,
                C=C, sigma_min=sigma_min, sigma_max=sigma_max, r_max=r_max, cover_n=cover_n, rtol=rtol, atol=atol, eps_factor=eps_factor)


In [ ]:
for r_max in r_max_list:
    for cover_n in cover_n_list:
        for rtol in rtol_list:
            row = compute_C(Z_test, model=model, R_fm=R_fm, r_max=r_max, cover_n=cover_n, rtol=rtol, atol=atol, eps_factor=eps_factor)
            row['timestamp'] = datetime.utcnow().isoformat()
            summary_rows.append(row)
            print(f"Done: r_max={r_max}, cover_n={cover_n}, rtol={rtol} -> C={row['C']}")


In [ ]:
import pandas as pd
df = pd.DataFrame(summary_rows)
csv_out = os.path.join(out_dir, f'convergence_summary_Z{Z_test}.csv')
df.to_csv(csv_out, index=False)
print('Saved summary to', csv_out)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,5))
for (cover_n, rtol), group in df.groupby(['cover_n','rtol']):
    ax.plot(group['r_max'], group['C'], marker='o', label=f"cover={cover_n}, rtol={rtol}")
ax.set_xlabel('r_max (a0)')
ax.set_ylabel('C(Z)')
ax.legend()
ax.grid(alpha=0.3)
plt.savefig(os.path.join(out_dir, f'convergence_plot_Z{Z_test}.png'), dpi=150)
plt.show()


## Next steps
- Inspect `results/convergence_summary_Z{Z_test}.csv` and the plot.
- If C stabilizes across refinements, record the parameter set as canonical and use it for production sweeps.
- If not stable, refine cover and tolerances further and repeat.
